In [5]:
import json
import pandas as pd

INPUT_JSON = "gen8ou.json"

# Output files
USAGE_CSV = "gen8ou_usage.csv"
ITEMS_CSV = "gen8ou_items.csv"
MOVES_CSV = "gen8ou_moves.csv"
TEAMMATES_CSV = "gen8ou_teammates.csv"
ALL_WIDE_CSV = "gen8ou_all.csv"

def clean_dict(d):
    """Remove junk keys and coerce values to float."""
    if not isinstance(d, dict):
        return {}
    out = {}
    for k, v in d.items():
        if k in ("", "empty", None):
            continue
        try:
            out[str(k)] = float(v)
        except Exception:
            continue
    return out

def flatten_any(value, prefix):
    """
    Flatten arbitrary JSON values into a flat dict of columns.
    - dict -> prefix:key columns (values numeric if possible, otherwise string/JSON)
    - list -> store as JSON string
    - scalar -> store as numeric if possible, else string
    """
    flat = {}

    if isinstance(value, dict):
        for k, v in value.items():
            col = f"{prefix}:{k}"
            if isinstance(v, (dict, list)):
                flat[col] = json.dumps(v)
            else:
                try:
                    flat[col] = float(v)
                except Exception:
                    flat[col] = str(v)

    elif isinstance(value, list):
        flat[prefix] = json.dumps(value)

    else:
        try:
            flat[prefix] = float(value)
        except Exception:
            flat[prefix] = str(value)

    return flat

with open(INPUT_JSON, "r", encoding="utf-8") as f:
    obj = json.load(f)

data = obj["data"]
total_battles = obj.get("info", {}).get("number of battles", 0) or 0

usage_rows = {}
items_rows = {}
moves_rows = {}
teammates_rows = {}
all_rows = {}

for mon, info in data.items():
    if not isinstance(info, dict):
        continue

    # --- usage fields ---
    usage = float(info.get("usage", 0) or 0)
    raw_usage = float(info.get("Raw count", 0) or 0)
    real_usage = (raw_usage / total_battles) if total_battles else 0.0

    usage_rows[mon] = {
        "usage": usage,
        "raw_usage": raw_usage,
        "real_usage": float(real_usage),
        "total_battles": float(total_battles)
    }

    # --- items/moves/teammates (NO prefixes for these individual datasets) ---
    items = clean_dict(info.get("Items", {}))
    moves = clean_dict(info.get("Moves", {}))
    teammates = clean_dict(info.get("Teammates", {}))

    items_rows[mon] = items
    moves_rows[mon] = moves
    teammates_rows[mon] = teammates

    # --- combined dataset (ALL DATA, flattened) ---
    combined = {}
    combined.update(usage_rows[mon])

    for top_key, top_val in info.items():
        # Skip the heavy / unwanted sections for the 5th dataset
        if top_key in ("Checks and Counters", "Spreads"):
            continue
        combined.update(flatten_any(top_val, top_key))

    all_rows[mon] = combined

# Build DataFrames (rows=pokemon, cols=features)
usage_df = pd.DataFrame.from_dict(usage_rows, orient="index").fillna(0)
items_df = pd.DataFrame.from_dict(items_rows, orient="index").fillna(0)
moves_df = pd.DataFrame.from_dict(moves_rows, orient="index").fillna(0)
teammates_df = pd.DataFrame.from_dict(teammates_rows, orient="index").fillna(0)
all_df = pd.DataFrame.from_dict(all_rows, orient="index")

# Ensure numeric + missing -> 0 for the first 4
for df in (usage_df, items_df, moves_df, teammates_df):
    df[:] = df.apply(pd.to_numeric, errors="coerce").fillna(0)

# For the big "all" file:
# - Convert columns that are actually numeric; keep text/JSON columns as strings
for c in all_df.columns:
    converted = pd.to_numeric(all_df[c], errors="coerce")
    if converted.notna().any():
        all_df[c] = converted.fillna(0)
    else:
        all_df[c] = all_df[c].fillna("")

# Add pokemon label once as first column
for df in (usage_df, items_df, moves_df, teammates_df, all_df):
    df.insert(0, "pokemon", df.index)

# Save
usage_df.to_csv(USAGE_CSV, index=False)
items_df.to_csv(ITEMS_CSV, index=False)
moves_df.to_csv(MOVES_CSV, index=False)
teammates_df.to_csv(TEAMMATES_CSV, index=False)
all_df.to_csv(ALL_WIDE_CSV, index=False)

print("✅ Wrote 5 files (rows=pokemon, cols=features; missing=0 or ''):")
print(" -", USAGE_CSV)
print(" -", ITEMS_CSV)
print(" -", MOVES_CSV)
print(" -", TEAMMATES_CSV)
print(" -", ALL_WIDE_CSV)
# this code was written using chatgpt to help convert json to csv files 
# Ignores usage stats for EV spreads and checks/counters, as they are not what I want to focus on for the graphs
# Five files to be able to mess around with them easier (5th file has all data aggregated)

C:\Users\Anath\AppData\Local\Temp\ipykernel_6468\2347557184.py:129: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df.insert(0, "pokemon", df.index)


✅ Wrote 5 files (rows=pokemon, cols=features; missing=0 or ''):
 - gen8ou_usage.csv
 - gen8ou_items.csv
 - gen8ou_moves.csv
 - gen8ou_teammates.csv
 - gen8ou_all.csv
